# REVIEWING DATASETS

In [ ]:
import pandas as pd
policy = pd.read_csv("../raw_data/sanlam_raw_policies.csv") 

In [ ]:
policy.head(10)

In [ ]:
policy.info()

In [ ]:
profiling = policy.nunique().reset_index()
profiling.columns = ["Column", "Distinct_Values"]
profiling

In [ ]:
policy['policy_status'].unique()

In [ ]:
policy.isnull().sum()

In [ ]:
policy.duplicated().sum()

In [ ]:
#Checking for consistency.
policy['sum_assured'].value_counts().head(20)

In [ ]:
#%pip install openpyxl

In [ ]:
clients = pd.read_excel("../raw_data/sanlam_raw_clients.csv.xlsx")

In [ ]:
clients.head(10)

In [ ]:
profiling = clients.nunique().reset_index()
profiling.columns = ["Column", "Distinct_Values"]
profiling

In [ ]:
clients.isnull().sum()

In [ ]:
clients.duplicated().sum()

In [ ]:
clients.info()

In [ ]:
#Checking for consistency.
clients['province'].value_counts().head(20)

# CLEANING DATASETS

In [ ]:
policy = pd.read_csv("../raw_data/sanlam_raw_policies.csv")
clients = pd.read_excel("../raw_data/sanlam_raw_clients.csv.xlsx")

In [ ]:
policy.head(5)

In [ ]:
policy.dtypes

In [ ]:
pd.DataFrame({
    'Column': policy.columns,
    'Data Type': policy.dtypes,
    'Non Null Count': policy.count().values,
    'Null Count': policy.isnull().sum().values
})

In [ ]:
missing_inception_date = policy[
    policy['inception_date'].isna() |
    policy['inception_date'].astype(str).str.strip().eq('')
].copy()

missing_inception_date

In [ ]:
policy[policy['policy_id'].duplicated(keep=False)] \
    .sort_values('policy_id')

In [ ]:
policy[
    policy['end_date'].isna()
][[
    'policy_id',
    'client_id',
    'product_code',
    'product_line',
    'end_date',
    'policy_status'
]].head(20)

In [ ]:
missing_ids = policy.loc[
    policy['inception_date'].isna(),
    'policy_id'
]

policy[
    policy['policy_id'].isin(missing_ids)
].sort_values('policy_id')

In [ ]:
quarantine = policy[
    policy['inception_date'].isna()
].copy()

quarantine['quarantine_reason'] = (
    'Missing inception_date'
)

In [ ]:
#Removing quarantined records.
policy_clean = policy[
    policy['inception_date'].notna()
].copy()

In [ ]:
policy_clean.to_csv(
    '../clean_data/policies_clean.csv',
    index=False
)

quarantine.to_csv(
    '../quarantine/inception_date_quarantine.csv',
    index=False
)

In [ ]:
quarantine = pd.read_csv("../quarantine/inception_date_quarantine.csv")
quarantine

In [ ]:
missing_end_date = policy[
    policy['end_date'].isna() |
    policy['end_date'].astype(str).str.strip().eq('')
].copy()

missing_end_date

In [160]:
policy[
    policy['end_date'].isna()
]['policy_status'].value_counts(dropna=False)

policy_status
In Force     21
Cancelled    11
Unknown      10
Lapsed        6
Active        4
Name: count, dtype: Int64

In [161]:
policy.head(20)

,source_row_id,policy_id,client_id,advisor_id,product_code,product_line,inception_date,end_date,annual_premium,sum_assured,policy_status,currency,source_system
0,SP-000000001,POL000000001,MISSING-763321,ADV000276,WEALTH,Wealth Investment,05/06/2022,04/06/2027,33254.23,"6.578.229,57",In Force,ZAR,WealthCore
1,SP-000000002,POL000000002,CL00011475,ADV000543,RA,Retirement Annuity,2021-09-05,2031/09/03,"ZAR 40,947.57","9,867,321.34",Lapsed,ZAR,WealthCore
2,SP-000000003,POL000000003,CL00004560,ADV000245,LIFE,Life Cover,2021/07/11,07-10-2026,46022.97,"R 7,841,284.09",In Force,ZAR,WealthCore
3,SP-000000004,POL000000004,CL00010896,ADV000588,WEALTH,Wealth Investment,07-23-2023,2033/07/20,"38.998,85","1,450,285.20",Unknown,R,LegacyLifeA
4,SP-000000005,POL000000005,CL00011016,ADV000019,RA,Retirement Annuity,2020-06-16,06-16-2021,"ZAR 21,239.96","R 1,396,051.35",Unknown,zar,WealthCore
5,SP-000000006,POL000000006,CL00003284,ADV000482,LIFE,Life Cover,11-06-2022,31/31/2024,"29.263,37","1.463.022,54",In Force,zar,LegacyLifeA
6,SP-000000007,POL000000007,CL00003297,ADV000489,LIFE,Life Cover,2019/03/15,2024-03-13,"39.059,96","10,957,534.54",Lapsed,ZAR,LegacyLifeA
7,SP-000000008,POL000000008,CL00009149,ADV000383,LIFE,Life Cover,18/06/2022,2027-06-17,"ZAR 19,717.92","5.226.636,62",Lapsed,zar,WealthCore
8,SP-000000009,POL000000009,CL00000846,ADV000518,LIFE,Life Cover,2022-06-16,2027/06/15,"ZAR 31,571.46","3,472,995.45",Lapsed,zar,LegacyLifeA
9,SP-000000010,POL000000010,CL00010149,ADV000500,LIFE,Life Cover,11-17-2022,2042/11/12,"44,001.26","R 10,854,659.95",Lapsed,ZAR,LegacyLifeA


In [166]:
# policy[
#     policy['end_date'].isna()
# ]['product_code'].value_counts(dropna=False)

policy['product_line'].unique()

array(['Wealth Investment', 'Retirement Annuity', 'Life Cover'],
      dtype=object)

In [165]:
# Standardise inconsistent product_line values
policy['product_line'] = (
    policy['product_line']
    .str.strip()
    .replace({
        'Wealth Invest.': 'Wealth Investment',
        'LIFE COVER': 'Life Cover',
        'Retirement annuity': 'Retirement Annuity'
    })
)


print(policy['product_line'].value_counts(dropna=False))

product_line
Retirement Annuity    8732
Wealth Investment     8683
Life Cover            8585
Name: count, dtype: int64


In [ ]:
policy['policy_status'] = (
    policy['policy_status']
    .astype('string')
    .str.strip()
    .replace({
        'INFORCE': 'In Force'
    })
)

In [ ]:
missing_annual_premium= policy[
    policy['annual_premium'].isna() |
    policy['annual_premium'].astype(str).str.strip().eq('')
].copy()

missing_annual_premium

In [ ]:
policy['inception_date'].dropna().head(20)

In [ ]:
# policy['inception_date'] = pd.to_datetime(
#     policy['inception_date'],
#     errors='coerce',
#     format='mixed'
# )

# policy['end_date'] = pd.to_datetime(
#     policy['end_date'],
#     errors='coerce',
#     format='mixed'
# )

In [ ]:
policy['end_date'].isnull()